# Setup

## Packages

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import json
import pandas as pd
from datetime import date

this_year = date.today().year

## Variables

In [ ]:
big_dict = {'director' : {
    'url' : 'https://challonge.com/bcmmmf26/predictions',
    'user' : {},
    'item_id' : {
        286619491: 'Martin Scorsese',
        286619608: 'Dennis Dugan',
        286619612: 'F. Gary Gray',
        286619615: 'Francois Truffaut',
        286619616: 'Anthony Minghella',
        286619617: 'Joe Johnston',
        286619618: 'James Wan',
        286619620: 'Hal Ashby',
        286619621: 'Warren Beatty',
        286619638: 'Andrei Tarkovsky',
        286619640: 'Penelope Spheeris',
        286619643: 'Dutch Verhoeven',
        286619646: 'Milos Forman',
        286619649: 'Bill Duke',
        286619651: 'Mira Nair',
        286619672: 'Judd Apatow',
        286619677: 'Wes Anderson',
        286619680: 'James Gray',
        286619682: 'Steve McQueen',
        286619685: 'Jacques Tati',
        286619779: 'Joe Wright',
        286619782: 'Curtis Hanson',
        286619783: 'Celine Sciamma',
        286619785: 'Preston Sturges',
        286619786: 'Tony Scott',
        286619787: 'Jia Zhangke',
        286619795: 'Mike Leigh',
        286619797: 'Karyn Kusama',
        286619789: 'Kelly Reichardt',
        286619793: 'Gus Van Sant',
        286619799: 'Oliver Stone',
        286619803: 'Chris Columbus'
        }
    }, 'patreon' : {
        'url' : 'https://challonge.com/bcmmpf26/predictions', 
        'user' : {},
        'item_id' : {
        286620164: 'Alvin and the Chipmunks',
        286620295 : 'Stand Up Animation',
        286620241 : 'Bridget Jones',
        286620274 : 'My Big Fat Greek Wedding',
        286620206 : 'Pauly Shore',
        286620277 : 'Cheech and Chong',
        286620204 : 'The Purge',
        286620290 : 'The Hunger Games',
        286620170 : 'Rambo',
        286620304 : 'Bourne',
        286620246 : 'Teen Shakespeare',
        286620275 : 'High School Musical',
        286620209 : 'Dirty Harry',
        286620279 : 'Bruce Lee',
        286620179 : 'Chucky',
        286620293 : 'Halloween',
        286620167 : 'Toxic Avenger',
        286620306 : 'Universal Soldier',
        286620239 : 'Airport',
        286620272 : 'Barbershop',
        286620236 : 'John Grisham',
        286620282 : 'Body Snatchers',
        286620185 : 'Cube',
        286620284 : 'Jumanji',
        286620166 : 'Lethal Weapon',
        286620307 : 'Neeson/Collet-Serra',
        286620261 : 'Oh God!',
        286620271 : 'Godzilla',
        286620210 : 'Major League',
        286620280 : 'Magic Mike',
        286620186 : 'Draculas',
        286620288 : 'Blair Witch'
        }
    }
}


# Scraping

In [ ]:
driver = webdriver.Chrome()

for each_one in big_dict.keys():  # "Each_one" = director or patreon

    # Get the main prediction page

    driver.get(big_dict[each_one]['url'])

    # Compile list of users

    predictors = [a.text for a 
                  in driver.find_elements(By.CSS_SELECTOR, 'span.prediction-link a')]
    
    # Compile list of links of their predictions

    predictor_links = [a.get_attribute('href') for a 
                       in driver.find_elements(By.CSS_SELECTOR, 'span.prediction-link a')]
    
    # Scrape each link. The two things we'll end up using are the predicted winner,
    # and the "data" list buried deep in the Javascript.
    # We'll then figure out how many wins the user predicted for each director/franchise by counting
    # the number of times the director/franchise's ID appears in the list. (Adding 1 for the predicted winner.)
    
    for user, link in zip(predictors, predictor_links):
    
        driver.get(link)

        big_dict[each_one]['user'][user] = {}

        big_dict[each_one]['user'][user]['prediction_attribute'] = driver.find_element(
            By.CSS_SELECTOR, 'div[data-react-props]').get_attribute("data-react-props")
        
        big_dict[each_one]['user'][user]['prediction_attribute'] = json.loads(big_dict[each_one]['user'][user]['prediction_attribute'])['initialPrediction']

        prediction_list = []

        for data_item in big_dict[each_one]['user'][user]['prediction_attribute']['data']:
                
            prediction_list.extend(data_item)

        big_dict[each_one]['user'][user]['predictions'] = prediction_list

        big_dict[each_one]['user'][user]['predicted_winner'] = big_dict[each_one]['user'][user]['prediction_attribute']['winner_id']

print(big_dict)


# Create final dictionary

 ## Each contest, user, and their # of wins predicted for each candidate

In [ ]:
predictions = {}

for each_one in big_dict.keys():
        
       predictions[each_one] = {}

       for user in big_dict[each_one]['user'].keys():
            
            predictions[each_one][user] = {}
            
            for id, referent in big_dict[each_one]['item_id'].items():

                if id == big_dict[each_one]['user'][user]['predicted_winner']:

                    predictions[each_one][user][referent] = big_dict[each_one]['user'][user]['predictions'].count(id) + 1

                else: 
            
                    predictions[each_one][user][referent] = big_dict[each_one]['user'][user]['predictions'].count(id)
                
print(predictions)


# Convert to dataframe and export

In [ ]:
for each_one in predictions.keys():

    df = pd.DataFrame.from_dict(predictions[each_one], orient = 'index')
    
    df.to_csv(f'{this_year}_blank_check_challonge_{each_one}.csv')
    